# 🔗 SQL 1 – Daten filtern, aggregieren und verknüpfen

## 🎯 Lernziele
Am Ende dieses Labs kannst du:
- die **Quellsysteme** der Sunny Bay Roastery und ihren Weg durch **Bronze → Silver → Gold** nachvollziehen
- Zeilen mit **WHERE** filtern (inklusive des Sonderfalls `NULL`)
- Daten mit **GROUP BY** zusammenfassen und Gruppen mit **HAVING** filtern
- Tabellen mit **INNER**, **LEFT**, **FULL OUTER**, **SEMI**, **ANTI** und **CROSS JOIN** verknüpfen – und einschätzen, wann welcher Join der richtige ist
- typische Join-Stolperfallen erkennen, etwa stillschweigend verlorene Zeilen und unsaubere Join-Schlüssel

## Einführung

**Warum dieses Lab?**

Ein Dashboard ist nur so gut wie der Gold Layer darunter – und ein guter Gold Layer entsteht größtenteils aus einer Handvoll SQL-Mustern: die richtigen Zeilen filtern, sie aggregieren und Daten aus verschiedenen Systemen miteinander verknüpfen. Genau diese Muster lernst du in diesem SQL Track anhand der Daten der Sunny Bay Roastery.

**Die Datenlandschaft von Sunny Bay**

Die Sunny Bay Roastery betreibt fünf Cafés (Hamburg, Berlin, München, Köln, Frankfurt am Main) und einen Onlineshop. Wie in jedem echten Unternehmen liegen die Daten in **mehreren operativen Systemen**. Jedes System liefert Rohdateien, die unverändert im **Bronze**-Layer landen:

| Quellsystem | Bronze-Tabellen | Inhalt |
|---|---|---|
| POS (Kassen der Cafés) | `pos_transactions`, `pos_transaction_lines` | Kassenbons der Cafés und die Produkte darauf |
| Webshop | `web_orders`, `web_accounts` | Online-Bestellungen (verschachteltes JSON) und Kundenkonten |
| CRM | `crm_customers`, `crm_loyalty_cards` | Kunden (mit Historienzeilen!) und Kundenkarten des Treueprogramms |
| ERP | `erp_products`, `erp_product_categories`, `erp_product_prices`, `erp_stores` | Stammdaten zu Produkten, Preisen und Filialen |
| HR | `hr_employees` | Mitarbeitende und ihre Vorgesetzten |
| Referenzdaten | `ref_public_holidays` | Gesetzliche Feiertage in Deutschland (die Cafés sind geschlossen) |

Der **Silver**-Layer bereinigt diese Tabellen und integriert sie zu **Business-Objekten**: `customer`, `loyalty_card`, `product`, `product_category`, `product_price`, `store`, `employee`, `public_holiday`, `sales_order` und `sales_order_line`.

Der **Gold**-Layer macht aus den Business-Objekten schließlich ein **Sternschema** für das Reporting: `fact_coffee_sales` mit den Dimensionen `dim_customer`, `dim_product`, `dim_store` und `dim_date`.

> **📝 Hinweis:** Die Daten umfassen ein rollierendes Fenster der **letzten 24 Monate** (sie werden erzeugt, wenn du Lab 0 ausführst). Deine Zahlen weichen deshalb leicht von denen deiner Sitznachbarn ab, wenn ihr den Workshop an unterschiedlichen Tagen eingerichtet habt. Die *Muster* in den Daten sind aber immer dieselben.

Dies ist das erste von drei SQL-Labs:

| Lab | Konzepte |
|---|---|
| **SQL 1 – Filtern, Aggregieren und Verknüpfen** (dieses Lab) | WHERE, GROUP BY, INNER / LEFT / FULL OUTER / SEMI / ANTI / CROSS JOIN |
| SQL 2 – Daten kombinieren und analysieren | UNION / MINUS / INTERSECT, WITH … AS, PARTITION BY, CUBE / ROLLUP, WITH RECURSIVE |
| SQL 3 – Funktionen für den Gold Layer | EXPLODE, QUALIFY, COALESCE, CASE / IFF, DECODE, MAX_BY, FILTER, DATEADD |

**So funktionieren die Übungen**

Jedes Konzept folgt demselben Rhythmus: eine kurze Erklärung, ein **Mini-Beispiel** mit ein paar handgemachten Zeilen, an dem du genau siehst, was passiert, ein Beispiel mit den Sunny-Bay-Daten und eine Übung **🏋️ Du bist dran**. Schreibe deine Antwort in die leere Zelle **🏋️ Du bist dran** unter der Übung und führe sie aus. Alle Lösungen findest du gesammelt im Notebook **SQL Lösungen** in diesem Ordner – versuche dich aber zuerst selbst an jeder Übung!

## Anleitung

Bevor du startest, prüfe bitte:
- Du hast **Lab 0 – Intro** abgeschlossen und der Setup-Job ist fertig (die Schemas `bronze`, `silver` und `gold` enthalten Tabellen).
- Dieses Notebook ist mit **Serverless** Compute oder einem **SQL warehouse** verbunden (z. B. *Serverless Starter Warehouse*).

**Schritt 1: Los geht's im Notebook**

1. Dieses Lab ist ein **SQL-Notebook**: Jede Code-Zelle kannst du direkt hier bearbeiten und ausführen. Verbinde das Notebook über die Auswahl oben rechts mit Compute – **Serverless** oder einem **SQL warehouse** wie *Serverless Starter Warehouse*.

2. Lege den Standard-Katalog fest. Alles, was du im SQL Track baust, landet im Schema `sql_sandbox`, das Lab 0 bereits für dich angelegt hat:

In [ ]:
USE CATALOG sunny_bay_germany;

> **❗ Wichtig:** Wenn du in Lab 0 einen anderen Katalognamen verwendet hast, ersetze `sunny_bay_germany` durch deinen Katalog. Wenn du ein **Präfix** verwendet hast, beginnen alle Tabellennamen damit (z. B. `silver.jd_customer` statt `silver.customer`). Wenn du dir den Katalog mit anderen Teilnehmenden teilst, setze dein Präfix auch vor die Tabellen, die du in `sql_sandbox` anlegst (z. B. `sql_sandbox.jd_demo_customer`).

3. Wirf einen ersten Blick auf die Silver-Tabellen:

In [ ]:
SHOW TABLES IN silver;

4. Öffne eine davon, um ein Gefühl für die Daten zu bekommen:

In [ ]:
SELECT * FROM silver.sales_order LIMIT 10;

> **✨ Tipp:** Führe die aktuelle Zelle mit **Shift + Enter** aus (führt sie aus und springt zur nächsten Zelle) oder mit **Ctrl + Enter** (**Cmd + Enter** auf dem Mac). Ändere ruhig jede Abfrage und führe sie erneut aus – Experimentieren ist ausdrücklich erwünscht! Verzichte auf **Run all**: Die Zellen „🏋️ Du bist dran“ sind leer (oder enthalten bei einem Startpunkt Lücken `___`), bis du sie ausfüllst.

**Schritt 2: Zeilen filtern mit WHERE**

`WHERE` behält nur die Zeilen, für die eine Bedingung wahr ist. Bedingungen lassen sich mit `AND` / `OR` kombinieren und verwenden Operatoren wie `=`, `<>`, `<`, `BETWEEN`, `IN`, `LIKE` und `IS NULL`.

1. Führe das Mini-Beispiel aus. Welche Zeilen erwartest du?

In [ ]:
SELECT *
FROM VALUES
    ('Single Espresso', 2.60, 'Drink'),
    ('Franzbrötchen',   3.20, 'Food'),
    ('House Blend 1kg', 32.00, 'Beans'),
    ('Iced Latte',      4.60, 'Drink')
    AS t(product, price_eur, category)
WHERE price_eur < 5
  AND category IN ('Drink', 'Food');

2. Das POS-Rohsystem markiert stornierte Bons mit dem Statuscode `V` (voided). Schauen wir sie uns an:

In [ ]:
SELECT transaction_id, store_id, transaction_ts, payment_code, status_code
FROM bronze.pos_transactions
WHERE status_code = 'V'
ORDER BY transaction_ts DESC;

> **📝 Hinweis:** Stornierte Bons dürfen nicht als Umsatz zählen. Deshalb behält die Silver-Tabelle `sales_order` nur Bons mit `status_code = 'C'` (completed, also abgeschlossen) – eine `WHERE`-Klausel ist oft der allererste Bereinigungsschritt.

3. Ein fehlender Wert heißt in SQL `NULL`. Im POS-System ist `loyalty_card_no` `NULL`, wenn der Gast keine Kundenkarte vorgezeigt hat. Zähle diese Bons:

In [ ]:
SELECT COUNT(*) AS receipts_without_card
FROM bronze.pos_transactions
WHERE loyalty_card_no IS NULL;

> **⚠️ Achtung:** `WHERE loyalty_card_no = NULL` liefert **überhaupt keine Zeilen**, denn ein Vergleich mit `NULL` ist nie wahr. Verwende immer `IS NULL` / `IS NOT NULL`.

4. 🏋️ **Du bist dran:** Finde alle Bestellungen in `silver.sales_order` aus dem **Café in Berlin** (`store_id = 2`), die in den **letzten 7 Tagen** mit **Girocard** bezahlt wurden. Zeige Bestell-ID, Zeitstempel und Zahlungsart, die neuesten zuerst.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 2 › Aufgabe 4*.

**Schritt 3: Zusammenfassen mit GROUP BY und HAVING**

`GROUP BY` fasst alle Zeilen mit denselben Werten zu **einer Zeile pro Gruppe** zusammen. Aggregatfunktionen wie `COUNT`, `SUM`, `AVG`, `MIN` und `MAX` berechnen dann einen Wert pro Gruppe. `HAVING` filtert die **Gruppen** nach der Aggregation – `WHERE` filtert die **Zeilen** davor.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    category,
    COUNT(*)       AS products,
    AVG(price_eur) AS avg_price_eur,
    MAX(price_eur) AS max_price_eur
FROM VALUES
    ('Single Espresso', 2.60, 'Drink'),
    ('Franzbrötchen',   3.20, 'Food'),
    ('House Blend 1kg', 32.00, 'Beans'),
    ('Iced Latte',      4.60, 'Drink')
    AS t(product, price_eur, category)
GROUP BY category;

2. Wie bezahlen die Kunden in den Cafés und online?

In [ ]:
SELECT
    sales_channel,
    payment_method,
    COUNT(*) AS orders
FROM silver.sales_order
GROUP BY sales_channel, payment_method
ORDER BY sales_channel, orders DESC;

> **✨ Tipp:** Databricks SQL unterstützt `GROUP BY ALL`: Damit wird automatisch nach allen ausgewählten Spalten gruppiert, die nicht aggregiert sind. `GROUP BY ALL` würde also dasselbe Ergebnis liefern wie `GROUP BY sales_channel, payment_method` oben.

3. Welche Produkte sind **Ladenhüter** mit weniger als 5.000 verkauften Einheiten? Hier bezieht sich die Bedingung auf ein Aggregat – deshalb gehört sie in `HAVING`:

In [ ]:
SELECT
    sku,
    SUM(quantity)                                            AS units_sold,
    ROUND(SUM(quantity * unit_price_eur - discount_eur), 2) AS gross_revenue_eur
FROM silver.sales_order_line
GROUP BY sku
HAVING SUM(quantity) < 5000
ORDER BY units_sold;

4. 🏋️ **Du bist dran:** Der CRM-Export `bronze.crm_customers` sollte eine Zeile pro Kunde enthalten – aber stimmt das auch? Finde alle `customer_id`s, die **mehr als einmal** vorkommen, und zeige, wie viele Zeilen jede davon hat.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 3 › Aufgabe 4*.

**Schritt 4: Zwei Mini-Tabellen für die Join-Beispiele anlegen**

Ein Join verknüpft Zeilen zweier Tabellen anhand einer **Join-Bedingung** (`ON …`). Damit du genau siehst, was jeder Join-Typ macht, verwenden wir zwei Mini-Tabellen: drei Kunden und drei Kundenkarten. Achte darauf: *Ben* hat keine Karte, und die Karte `C-9` gehört zu einem Kunden, den es gar nicht gibt.

1. Lege die beiden Demo-Tabellen in deinem Schema `sql_sandbox` an:

In [ ]:
CREATE OR REPLACE TABLE sql_sandbox.demo_customer AS
SELECT * FROM VALUES
    (1, 'Anna'),
    (2, 'Ben'),
    (3, 'Clara')
    AS t(customer_id, name);

CREATE OR REPLACE TABLE sql_sandbox.demo_card AS
SELECT * FROM VALUES
    ('C-1', 1, 'Gold'),
    ('C-3', 3, 'Silver'),
    ('C-9', 9, 'Bronze')
    AS t(card_no, customer_id, tier);

2. Behalte diese Übersicht griffbereit – sie fasst die Joins dieses Labs zusammen:

| Join | Behält … | Typische Frage |
|---|---|---|
| `INNER JOIN` | nur Zeilen mit einem Treffer auf **beiden** Seiten | „Positionen **mit** ihrem Produkt“ |
| `LEFT JOIN` | **alle** Zeilen der linken Tabelle, dazu die Treffer von rechts (oder `NULL`) | „Alle Kunden und ihre Karte, **falls vorhanden**“ |
| `FULL OUTER JOIN` | **alle** Zeilen **beider** Tabellen | „Zwei Systeme vergleichen: Wer ist in A, wer in B, wer in beiden?“ |
| `LEFT SEMI JOIN` | linke Zeilen, die einen Treffer **haben** (nur linke Spalten, keine Duplikate) | „Produkte, die **jemals** online verkauft wurden“ |
| `LEFT ANTI JOIN` | linke Zeilen **ohne** Treffer | „Mitglieder, die **nie** etwas gekauft haben“ |
| `CROSS JOIN` | **jede** Kombination beider Tabellen | „Jede Filiale × jeder Tag“ |

> **✨ Tipp:** **Zwei Stilregeln, die wir im ganzen Workshop verwenden:** Schreibe den Join-Typ immer explizit aus (`INNER JOIN` statt eines einfachen `JOIN` – beides bedeutet dasselbe, aber die explizite Form liest sich neben `LEFT JOIN` & Co. leichter), und führe einen Tabellenalias immer mit `AS` ein (`FROM silver.product AS p`). Beides ist in SQL optional, macht Abfragen aber deutlich leichter zu lesen und zu prüfen.

**Schritt 5: INNER JOIN – nur passende Zeilen**

1. Führe das Mini-Beispiel aus. Ben (keine Karte) und die Karte `C-9` (kein Kunde) verschwinden:

In [ ]:
SELECT c.customer_id, c.name, k.card_no, k.tier
FROM sql_sandbox.demo_customer AS c
INNER JOIN sql_sandbox.demo_card AS k
    ON c.customer_id = k.customer_id;

2. Bestellpositionen enthalten nur eine `product_id`. Verknüpfe sie mit den Produktstammdaten, um zu sehen, was verkauft wurde:

In [ ]:
SELECT l.order_id, l.line_no, p.product_name, l.quantity, l.unit_price_eur
FROM silver.sales_order_line AS l
INNER JOIN silver.product AS p
    ON l.product_id = p.product_id
LIMIT 20;

3. Ein Inner Join **verwirft stillschweigend** Zeilen ohne Treffer. Vergleiche die Anzahl der rohen POS-Positionen mit der Anzahl der Positionen, die über die SKU ihr Produkt finden:

In [ ]:
SELECT
    (SELECT COUNT(*) FROM bronze.pos_transaction_lines)  AS all_pos_lines,
    (SELECT COUNT(*)
     FROM bronze.pos_transaction_lines AS l
     INNER JOIN silver.product AS p
         ON l.sku = p.sku)                                AS lines_with_known_product;

> **📝 Hinweis:** Ein paar Positionen gehen verloren. Sie gehören zur SKU `SB-9999` – einem Testartikel, den Auszubildende an der Kasse gescannt haben. In diesem Fall ist das Verwerfen genau das, was wir wollen. Prüfe aber immer die Zeilenanzahl vor und nach einem Join: Ein unerwarteter Unterschied deutet meist auf ein Problem mit der Datenqualität hin.

4. 🏋️ **Du bist dran:** Berechne den **Bruttoumsatz pro Produktname** (`quantity * unit_price_eur - discount_eur`) und zeige die 10 umsatzstärksten Produkte.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 5 › Aufgabe 4*.

**Schritt 6: LEFT JOIN – alles von links behalten**

1. Führe das Mini-Beispiel aus. Ben bleibt erhalten, aber seine Kartenspalten sind `NULL`:

In [ ]:
SELECT c.customer_id, c.name, k.card_no, k.tier
FROM sql_sandbox.demo_customer AS c
LEFT JOIN sql_sandbox.demo_card AS k
    ON c.customer_id = k.customer_id;

2. Zeige jeden Kunden zusammen mit seiner Kundenkarte – falls er eine hat:

In [ ]:
SELECT c.customer_id, c.first_name, c.last_name, lc.card_no, lc.loyalty_tier, lc.is_active
FROM silver.customer AS c
LEFT JOIN silver.loyalty_card AS lc
    ON c.customer_id = lc.customer_id
ORDER BY c.customer_id
LIMIT 50;

> **⚠️ Achtung:** **Join-Fan-out (Zeilenvervielfachung):** Hat die rechte Tabelle *mehrere* Treffer, wird die linke Zeile wiederholt. Manche Kunden haben ihre Karte verloren und eine neue bekommen – deshalb tauchen sie oben zweimal auf. Dasselbe passiert, wenn du den rohen CRM-Export mit seinen Historienzeilen verknüpfst: Jeder Kunde mit zwei Versionen würde in deinem Umsatz plötzlich doppelt zählen. Kenne vor jedem Join immer die **Granularität** (was eine Zeile bedeutet) beider Tabellen.

3. 🏋️ **Du bist dran:** Wie viele Kunden in `silver.customer` hatten **noch nie** eine Kundenkarte? Verwende einen `LEFT JOIN` und suche nach den `NULL`s.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 6 › Aufgabe 3*.

**Schritt 7: FULL OUTER JOIN – zwei Systeme vergleichen**

1. Führe das Mini-Beispiel aus. Du bekommst Anna, Ben (ohne Karte), Clara **und** die verwaiste Karte `C-9` (ohne Kunde):

In [ ]:
SELECT c.customer_id, c.name, k.card_no, k.customer_id AS card_customer_id
FROM sql_sandbox.demo_customer AS c
FULL OUTER JOIN sql_sandbox.demo_card AS k
    ON c.customer_id = k.customer_id;

2. Man kann Kunde in den Cafés (CRM), im Onlineshop (Webshop) – oder in beiden sein. Das Einzige, was die beiden Systeme gemeinsam haben, ist die **E-Mail-Adresse**. Ein Full Outer Join zeigt alle drei Gruppen:

In [ ]:
SELECT
    CASE
        WHEN crm.customer_id IS NULL THEN 'Webshop only'
        WHEN web.account_id IS NULL  THEN 'CRM only'
        ELSE 'Both systems'
    END      AS found_in,
    COUNT(*) AS row_count
FROM bronze.crm_customers AS crm
FULL OUTER JOIN bronze.web_accounts AS web
    ON LOWER(TRIM(crm.email)) = web.email
GROUP BY 1
ORDER BY 1;

3. 🏋️ **Du bist dran:** Warum vergleichen wir `LOWER(TRIM(crm.email))` statt einfach `crm.email`? Zähle die passenden Zeilen (`INNER JOIN`) einmal **mit** und einmal **ohne** `LOWER(TRIM(...))`. Sieh dir dann einige CRM-E-Mail-Adressen an, die nicht schon kleingeschrieben sind.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 7 › Aufgabe 3*.

**Schritt 8: LEFT SEMI JOIN – „hat mindestens einen Treffer“**

Ein Semi Join behält die Zeilen der linken Tabelle, die **mindestens einen** Treffer in der rechten Tabelle haben. Anders als ein Inner Join liefert er nur die linken Spalten und **dupliziert nie** eine Zeile, egal wie viele Treffer es gibt. In Standard-SQL schreibst du dasselbe mit `WHERE EXISTS (…)`.

1. Führe das Mini-Beispiel aus. Nur Anna und Clara bleiben übrig – ohne Kartenspalten:

In [ ]:
SELECT *
FROM sql_sandbox.demo_customer AS c
LEFT SEMI JOIN sql_sandbox.demo_card AS k
    ON c.customer_id = k.customer_id;

2. Welche Produkte wurden mindestens einmal **online** verkauft?

In [ ]:
SELECT p.product_id, p.product_name
FROM silver.product AS p
LEFT SEMI JOIN (
    SELECT l.product_id
    FROM silver.sales_order_line AS l
    INNER JOIN silver.sales_order AS o
        ON l.order_id = o.order_id
    WHERE o.sales_channel = 'Online'
) AS online_lines
    ON p.product_id = online_lines.product_id
ORDER BY p.product_id;

3. Dieselbe Frage, geschrieben mit `EXISTS`:

In [ ]:
SELECT p.product_id, p.product_name
FROM silver.product AS p
WHERE EXISTS (
    SELECT 1
    FROM silver.sales_order_line AS l
    INNER JOIN silver.sales_order AS o
        ON l.order_id = o.order_id
    WHERE o.sales_channel = 'Online'
      AND l.product_id = p.product_id
)
ORDER BY p.product_id;

> **✨ Tipp:** Ersetze in Abfrage 2 testweise `LEFT SEMI JOIN` durch `INNER JOIN` (und wähle `p.*` aus). Dann bekommst du jedes Produkt tausendfach – einmal pro Online-Bestellposition. Verwende einen Semi Join immer dann, wenn du nur nach dem Vorhandensein zugehöriger Zeilen **filtern** willst.

4. 🏋️ **Du bist dran:** Welche Kundenkarten wurden in den **letzten 30 Tagen** für mindestens einen Einkauf im Café verwendet? Zeige Kartennummer, Kunde und Stufe.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 8 › Aufgabe 4*.

**Schritt 9: LEFT ANTI JOIN – „hat keinen Treffer“**

Ein Anti Join ist das Gegenteil eines Semi Joins: Er behält die linken Zeilen, die **überhaupt keinen** Treffer haben. In Standard-SQL schreibst du ihn mit `WHERE NOT EXISTS (…)` oder mit dem Muster „LEFT JOIN + IS NULL“ aus Schritt 6.

1. Führe das Mini-Beispiel aus. Nur Ben bleibt übrig:

In [ ]:
SELECT *
FROM sql_sandbox.demo_customer AS c
LEFT ANTI JOIN sql_sandbox.demo_card AS k
    ON c.customer_id = k.customer_id;

2. Das Marketingteam möchte Mitglieder des Treueprogramms zurückgewinnen, die sich zwar angemeldet, aber mit ihrer Karte **noch nie etwas gekauft** haben:

In [ ]:
SELECT lc.card_no, lc.customer_id, lc.loyalty_tier, lc.enrolled_on
FROM silver.loyalty_card AS lc
LEFT ANTI JOIN silver.sales_order AS o
    ON lc.card_no = o.loyalty_card_no
WHERE lc.is_active
ORDER BY lc.enrolled_on DESC;

3. 🏋️ **Du bist dran:** Welche Produkte aus den Produktstammdaten wurden **noch nie verkauft**? Schreibe die Abfrage einmal mit `LEFT ANTI JOIN` und einmal mit `NOT EXISTS`. Vergleiche dein Ergebnis anschließend mit der Spalte `has_sales` von `gold.dim_product`.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 9 › Aufgabe 3*.

**Schritt 10: CROSS JOIN – jede Kombination**

Ein Cross Join hat **keine Join-Bedingung**: Er kombiniert jede Zeile der linken Tabelle mit jeder Zeile der rechten Tabelle. Das klingt gefährlich (und ist es auch – 1.000 × 1.000 Zeilen sind bereits eine Million Zeilen), aber genau damit baust du **vollständige Raster**, z. B. „jede Filiale an jedem Tag“, die du anschließend mit den tatsächlichen Daten vergleichen kannst.

1. Führe das Mini-Beispiel aus – jedes Getränk in jeder Größe:

In [ ]:
SELECT d.drink, s.size
FROM VALUES ('Latte'), ('Cappuccino') AS d(drink)
CROSS JOIN VALUES ('Small'), ('Medium'), ('Large') AS s(size);

2. Baue das Raster aus jeder Filiale und jedem Tag des Verkaufszeitraums und behalte die Kombinationen **ohne einen einzigen Verkauf** (der Anti Join aus Schritt 9):

In [ ]:
SELECT s.store_name, d.date, d.day_name, d.holiday_name
FROM silver.store AS s
CROSS JOIN gold.dim_date AS d
LEFT ANTI JOIN silver.sales_order AS o
    ON  o.store_id = s.store_id
    AND o.order_date = d.date
WHERE d.date BETWEEN (SELECT MIN(order_date) FROM silver.sales_order)
                 AND (SELECT MAX(order_date) FROM silver.sales_order)
  AND d.date >= s.opened_on
ORDER BY d.date, s.store_name;

> **📝 Hinweis:** Alle Tage ohne Verkäufe sind gesetzliche **Feiertage**, an denen die Cafés geschlossen sind. Der Onlineshop taucht nie auf – er verkauft an jedem Tag. Ohne den Cross Join könntest du diese „fehlenden“ Zeilen nie sehen, denn ein Tag ohne Verkäufe hat in der Umsatztabelle schlicht keine Zeile.

3. 🏋️ **Du bist dran:** Wie viele Zeilen hat das Raster **vor** dem Anti Join? Zähle die Filialen, die Tage des Verkaufszeitraums und die Zeilen des Cross Joins und prüfe, ob Zeilen = Filialen × Tage gilt.

In [ ]:
-- 🏋️ Du bist dran: schreibe hier deine Abfrage und führe die Zelle aus

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 1 › Schritt 10 › Aufgabe 3*.

**Schritt 11: Eine echte Gold-Tabelle ansehen**

1. Öffne die Definition der Gold-Faktentabelle in der Pipeline: Navigiere im Workspace-Browser zu deinem Git-Ordner → `bundle/src/transformations/gold/fact_coffee_sales.sql`. Fast alles darin kennst du jetzt aus diesem Lab:
   - `INNER JOIN` von Bestellpositionen, Bestellungen und Produkten
   - `LEFT JOIN` zur Kundenkarte und zum Webshop-Kundenkonto, um den Kunden zu finden
   - ein `LEFT JOIN` mit einer **Bereichsbedingung** (`BETWEEN valid_from AND valid_to`), um den Preis auszuwählen, der am Bestelldatum gültig war

> **✨ Tipp:** Der Range Join ist ein sehr häufiges Muster im Gold Layer – für alles, was sich im Lauf der Zeit ändert: Preise, Wechselkurse, Steuersätze oder die Vertriebsregion eines Kunden. Die Join-Bedingung muss also keine Gleichheit sein!

## Wie geht es weiter?

🎉 Gut gemacht! Du kannst jetzt Daten filtern, aggregieren und – vor allem – Daten aus verschiedenen Quellsystemen verknüpfen. Und du kennst die klassischen Stolperfallen: verlorene Zeilen, Fan-out und unsaubere Join-Schlüssel.

In **SQL 2 – Daten kombinieren und analysieren** stapelst du Ergebnismengen mit `UNION`, `MINUS` und `INTERSECT`, strukturierst lange Abfragen mit `WITH`, analysierst Daten mit Window-Funktionen (`PARTITION BY`), baust Zwischensummen mit `ROLLUP` und `CUBE` und durchläufst Hierarchien mit `WITH RECURSIVE`.